In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# ── Wczytanie danych ──────────────────────────────────────────────────────
df_agent = pd.read_csv("agent_candidates_admet.csv")
df_novel = pd.read_csv("novel_molecules_admet.csv")

print(f"agent_candidates_admet.csv: {len(df_agent)} związków")
print(f"novel_molecules_admet.csv:  {len(df_novel)} związków")

agent_candidates_admet.csv: 1687 związków
novel_molecules_admet.csv:  402 związków


In [3]:
# ── Definicje właściwości ────────────────────────────────────────────────
# (kolumna, etykieta, jednostka/skala, próg bezpieczeństwa)
PROPERTIES = [
    ("caco2",      "Caco-2 (przepuszczalność, log)",        None,        None),
    ("bbb_prob",   "BBB (P przenikania)",                    (0, 1),      0.5),
    ("cyp3a4_prob","CYP3A4 (P inhibicji)",                   (0, 1),      0.5),
    ("half_life",  "Half-life [h]",                          None,        None),
    ("herg_prob",  "hERG (P toksyczności)",                  (0, 1),      0.5),
    ("hum_mic_cl", "Human microsomal clearance (log)",       None,        None),
    ("ppb",        "PPB (frakcja związana)",                 (0, 1.05),   None),
    ("logd",       "LogD (lipofilowość)",                    None,        None),
    ("water_sol",  "Rozpuszczalność wodna (logS)",           None,        None),
]

THRESHOLDS = {
    "herg_prob":   ("below", 0.5, "P(hERG) < 0.5 — bezpieczny kardiologicznie"),
    "cyp3a4_prob": ("below", 0.5, "P(CYP3A4) < 0.5 — brak silnej inhibicji"),
}

In [11]:
# ── Funkcja: dashboard rozkładów ADMET ──────────────────────────────────────
def plot_admet_distributions(df: pd.DataFrame, title: str, fname: str):
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()

    for i, (col, label, xlim, threshold) in enumerate(PROPERTIES):
        ax = axes[i]
        data = df[col].dropna()

        ax.hist(data, bins=30, color="#534AB7", alpha=0.75, edgecolor="white")

        if threshold is not None:
            ax.axvline(threshold, color="#D85A30", linestyle="--", linewidth=1.5,
                       label=f"próg = {threshold}")
            ax.legend(fontsize=8)

        if xlim is not None:
            ax.set_xlim(xlim)

        ax.set_title(label, fontsize=11)
        ax.set_xlabel("wartość")
        ax.set_ylabel("liczba związków")

        # statystyki w tekście
        stats_text = f"n={len(data)}\nmean={data.mean():.2f}\nmedian={data.median():.2f}"
        ax.text(0.97, 0.95, stats_text, transform=ax.transAxes,
                ha="right", va="top", fontsize=8,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray"))

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Zapisano: {fname}")


In [12]:
# ── Funkcja: applicability domain ───────────────────────────────────────────
def plot_applicability_domain(df: pd.DataFrame, title: str, fname: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # 1. Histogram tanimoto_max
    ax = axes[0]
    ax.hist(df["tanimoto_max"].dropna(), bins=30, color="#1D9E75", alpha=0.75, edgecolor="white")
    ax.axvline(0.4, color="#D85A30", linestyle="--", linewidth=1.5, label="próg = 0.4")
    ax.set_title("Tanimoto similarity do zbioru treningowego")
    ax.set_xlabel("max Tanimoto similarity")
    ax.set_ylabel("liczba związków")
    ax.legend()

    # 2. Pie chart in_domain
    ax = axes[1]
    counts = df["in_domain"].value_counts()
    labels = []
    values = []
    colors = []
    for val, color in [(True, "#1D9E75"), (False, "#D85A30")]:
        if val in counts.index:
            n = counts[val]
            pct = 100 * n / len(df)
            labels.append(f"{'W domenie' if val else 'Poza domeną'}\n(n={n}, {pct:.1f}%)")
            values.append(n)
            colors.append(color)

    ax.pie(values, labels=labels, colors=colors, autopct="", startangle=90,
           wedgeprops={"edgecolor": "white", "linewidth": 1.5})
    ax.set_title("Applicability domain")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Zapisano: {fname}")

In [13]:
# ── Funkcja: podsumowanie filtra ADMET ──────────────────────────────────────
def plot_safety_summary(df: pd.DataFrame, title: str, fname: str):
    n_total = len(df)

    # Definicje progów
    n_safe_herg  = (df["herg_prob"] < 0.5).sum()
    n_safe_cyp   = (df["cyp3a4_prob"] < 0.5).sum()
    n_in_domain  = df["in_domain"].sum()
    n_all_pass   = (
        (df["herg_prob"] < 0.5) &
        (df["cyp3a4_prob"] < 0.5) &
        (df["in_domain"] == True)
    ).sum()

    categories = [
        "Wszystkie\nzwiązki",
        "hERG\nbezpieczny\n(P<0.5)",
        "CYP3A4\nbez inhibicji\n(P<0.5)",
        "W domenie\n(Tanimoto≥0.4)",
        "Przeszły\nWSZYSTKO",
    ]
    values = [n_total, n_safe_herg, n_safe_cyp, n_in_domain, n_all_pass]
    colors = ["#5F5E5A", "#378ADD", "#378ADD", "#1D9E75", "#0F6E56"]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    bars = ax.bar(categories, values, color=colors, edgecolor="white")

    for bar, val in zip(bars, values):
        pct = 100 * val / n_total
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_total*0.01,
               f"{val}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)

    ax.set_ylabel("liczba związków")
    ax.set_ylim(0, n_total * 1.15)
    ax.set_title(title, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Zapisano: {fname}")

In [16]:
# ── Funkcja: korelacja MW/QED vs niepewność (jeśli kolumny istnieją) ────────
def plot_uncertainty_overview(df: pd.DataFrame, title: str, fname: str):
    """Pokazuje średnią niepewność (std) MC Dropout dla każdej właściwości."""
    std_cols = [c for c in df.columns if c.endswith("_std")]
    rel_uncertainty = {}
    for std_col in std_cols:
        base_col = std_col.replace("_std", "")
        value_range = df[base_col].max() - df[base_col].min()
        if value_range == 0:
            continue
        # std jako % zakresu wartości
        rel_uncertainty[base_col] = (df[std_col] / value_range).mean() * 100

    labels = list(rel_uncertainty.keys())
    values = list(rel_uncertainty.values())

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(labels, values, color="#D4537E", edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(val + max(values)*0.01, bar.get_y() + bar.get_height()/2,
               f"{val:.1f}%", va="center", fontsize=9)

    ax.set_xlabel("średnia niepewność jako % zakresu wartości")
    ax.set_title(title, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Zapisano: {fname}")

In [18]:
def plot_classification_uncertainty(df: pd.DataFrame, title: str, fname: str):
    cls_props = ["bbb_prob", "cyp3a4_prob", "herg_prob"]
    means = [df[f"{p}_std"].mean() for p in cls_props]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(cls_props, means, color="#378ADD", edgecolor="white")
    for bar, val in zip(bars, means):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
               f"{val:.3f}", va="center", fontsize=9)
    ax.set_xlabel("średnia niepewność (std prawdopodobieństwa)")
    ax.set_title(title, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Zapisano: {fname}")

In [19]:
# ── Generowanie wszystkich wykresów ─────────────────────────────────────────
print("\n=== agent_candidates_admet.csv ===")
plot_admet_distributions(df_agent, "Rozkłady właściwości ADMET — Agent candidates", "agent_admet_distributions.png")
plot_applicability_domain(df_agent, "Applicability domain — Agent candidates", "agent_applicability_domain.png")
plot_safety_summary(df_agent, "Filtr bezpieczeństwa ADMET — Agent candidates", "agent_safety_summary.png")
plot_uncertainty_overview(df_agent, "Niepewność predykcji (MC Dropout) — Agent candidates", "agent_uncertainty.png")
plot_classification_uncertainty(df_agent, "Niepewność predykcji dla zadań klasyfikacyjnych — Agent candidates", "agent_classification_uncertainty.png")


=== agent_candidates_admet.csv ===
  Zapisano: agent_admet_distributions.png
  Zapisano: agent_applicability_domain.png
  Zapisano: agent_safety_summary.png
  Zapisano: agent_classification_uncertainty.png


In [21]:
print("\n=== novel_molecules_admet.csv ===")
plot_admet_distributions(df_novel, "Rozkłady właściwości ADMET — Novel molecules", "novel_admet_distributions.png")
plot_applicability_domain(df_novel, "Applicability domain — Novel molecules", "novel_applicability_domain.png")
plot_safety_summary(df_novel, "Filtr bezpieczeństwa ADMET — Novel molecules", "novel_safety_summary.png")
plot_uncertainty_overview(df_novel, "Niepewność predykcji (MC Dropout) — Novel molecules", "novel_uncertainty.png")
plot_classification_uncertainty(df_novel, "Niepewność predykcji dla zadań klasyfikacyjnych — Novel candidates", "novel_classification_uncertainty.png")


=== novel_molecules_admet.csv ===
  Zapisano: novel_admet_distributions.png
  Zapisano: novel_applicability_domain.png
  Zapisano: novel_safety_summary.png
  Zapisano: novel_classification_uncertainty.png


In [10]:
# ── Tekstowe podsumowanie ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PODSUMOWANIE")
print("=" * 60)

for name, df in [("agent_candidates", df_agent), ("novel_molecules", df_novel)]:
    n = len(df)
    print(f"\n--- {name} ({n} związków) ---")
    print(f"  W domenie (Tanimoto >= 0.4):  {df['in_domain'].sum():5d} ({100*df['in_domain'].sum()/n:.1f}%)")
    print(f"  hERG bezpieczny (P<0.5):       {(df['herg_prob']<0.5).sum():5d} ({100*(df['herg_prob']<0.5).sum()/n:.1f}%)")
    print(f"  CYP3A4 brak inhibicji (P<0.5): {(df['cyp3a4_prob']<0.5).sum():5d} ({100*(df['cyp3a4_prob']<0.5).sum()/n:.1f}%)")
    print(f"  BBB penetruje (P>0.5):         {(df['bbb_prob']>0.5).sum():5d} ({100*(df['bbb_prob']>0.5).sum()/n:.1f}%)")
    all_pass = ((df['herg_prob']<0.5) & (df['cyp3a4_prob']<0.5) & (df['in_domain']==True)).sum()
    print(f"  PRZESZŁY WSZYSTKIE PROGI:       {all_pass:5d} ({100*all_pass/n:.1f}%)")

    if "verdict" in df.columns:
        print(f"  Werdykt agenta: {df['verdict'].value_counts().to_dict()}")



PODSUMOWANIE

--- agent_candidates (1687 związków) ---
  W domenie (Tanimoto >= 0.4):   1012 (60.0%)
  hERG bezpieczny (P<0.5):         269 (15.9%)
  CYP3A4 brak inhibicji (P<0.5):   979 (58.0%)
  BBB penetruje (P>0.5):          1489 (88.3%)
  PRZESZŁY WSZYSTKIE PROGI:         128 (7.6%)

--- novel_molecules (402 związków) ---
  W domenie (Tanimoto >= 0.4):    148 (36.8%)
  hERG bezpieczny (P<0.5):          86 (21.4%)
  CYP3A4 brak inhibicji (P<0.5):   209 (52.0%)
  BBB penetruje (P>0.5):           300 (74.6%)
  PRZESZŁY WSZYSTKIE PROGI:          25 (6.2%)
  Werdykt agenta: {'PASS': 402}
